# Dimension Educacion y Desarrollo - Calculo para el ITT

**Objetivo:** Calcular el score de la dimension Educacion y Desarrollo del ITT
usando los datos reales 2026 del Observatorio de Educacion de Cali.

**Indicadores usados:**
- Matricula escolar por comuna (dato real)
- Tasa de repitencia (dato municipal como referente)
- Estudiantes por docente (dato municipal)
- Estudiantes por equipo de computo (dato por sede, agregable a comuna)

**Metodologia:** Normalizacion con ref_min/ref_max fijos, promedio simple de indicadores,
score de dimension en escala 0-100.

## Celda 0 - Configuracion

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/Observatorio_de_Educacion'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/j0rg3c45/Observatorio_de_Educacion.git ' + REPO_DIR)
    os.chdir(REPO_DIR)
    DATA_PATH = 'data/Fuentes de datos'
else:
    DATA_PATH = '../data/Fuentes de datos'
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'Datos: {DATA_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.float_format', '{:,.2f}'.format)

def fmt_num(v):
    if abs(v) >= 1_000_000: return f'{v/1_000_000:.1f}M'
    elif abs(v) >= 1_000: return f'{v/1_000:.1f}K'
    else: return f'{v:,.0f}'

def fmt_eje(ax, eje='x'):
    def _f(x, pos):
        if abs(x)>=1e6: return f'{x/1e6:.1f}M'
        elif abs(x)>=1e3: return f'{x/1e3:.0f}K'
        else: return f'{x:.0f}'
    if eje=='x': ax.xaxis.set_major_formatter(mticker.FuncFormatter(_f))
    else: ax.yaxis.set_major_formatter(mticker.FuncFormatter(_f))

print('Listo')

---
## 1. Definicion de ref_min / ref_max para indicadores educativos

Basados en estandares MEN, metas del Plan de Desarrollo y datos observados 2026.

In [ ]:
# ============================================================
# REFERENCIAS FIJAS - DIMENSION EDUCACION Y DESARROLLO
# ============================================================
# Formato: (ref_min, ref_max, inverso, descripcion)
#
# ref_min = mejor resultado razonable (aspiracional)
# ref_max = peor resultado tolerable
# inverso = True si menor valor es mejor
# ============================================================

REFS_EDUCACION = {
    'matricula': {
        'ref_min': 5000,
        'ref_max': 30000,
        'inverso': False,
        'desc': 'Matricula total por comuna (mayor = mas acceso)'
    },
    'repitencia': {
        'ref_min': 1.0,
        'ref_max': 15.0,
        'inverso': True,
        'desc': 'Tasa de repitencia % (menor = mejor)'
    },
    'est_por_docente': {
        'ref_min': 15.0,
        'ref_max': 40.0,
        'inverso': True,
        'desc': 'Estudiantes por docente (menor = mejor atencion)'
    },
    'est_por_equipo': {
        'ref_min': 1.0,
        'ref_max': 20.0,
        'inverso': True,
        'desc': 'Estudiantes por equipo computo (menor = mejor dotacion)'
    },
}

print('Referencias definidas:')
for k, v in REFS_EDUCACION.items():
    tipo = 'INVERSO' if v['inverso'] else 'POSITIVO'
    print(f"  {k:20s} [{v['ref_min']:6.1f} - {v['ref_max']:6.1f}] {tipo} | {v['desc']}")

---
## 2. Funcion de normalizacion

In [ ]:
def score_ref(valor, ref_min, ref_max, inverso):
    """Normaliza un valor con umbrales fijos ref_min/ref_max.
    Retorna score entre 0 y 100."""
    if ref_max == ref_min:
        return 100.0
    raw = np.clip((valor - ref_min) / (ref_max - ref_min) * 100, 0, 100)
    return 100 - raw if inverso else raw

# Verificacion rapida
print('Verificacion de la funcion:')
print(f'  Matricula 25,000 -> score: {score_ref(25000, 5000, 30000, False):.1f} (esperado ~80)')
print(f'  Repitencia 6.82% -> score: {score_ref(6.82, 1.0, 15.0, True):.1f} (esperado ~58)')
print(f'  Est/docente 24.6 -> score: {score_ref(24.6, 15.0, 40.0, True):.1f} (esperado ~62)')
print(f'  Est/equipo 4.88  -> score: {score_ref(4.88, 1.0, 20.0, True):.1f} (esperado ~80)')

---
## 3. Cargar datos y construir tabla por comuna

In [ ]:
# --- Matricula por comuna ---
ruta_mat = os.path.join(DATA_PATH, 'Reporte de matr\u00edcula', '01_Matricula_2026.xlsx')
if not os.path.exists(ruta_mat):
    ruta_mat = os.path.join(DATA_PATH, 'Reporte de matricula', '01_Matricula_2026.xlsx')
df_comuna = pd.read_excel(ruta_mat, sheet_name='Por comuna')
df_cu = df_comuna[df_comuna['comuna'].str.contains('Comuna', na=False)].copy()
df_cu['num_comuna'] = df_cu['comuna'].str.extract(r'(\d+)').astype(int)
df_cu = df_cu.sort_values('num_comuna').reset_index(drop=True)

# --- Detalle por sede (equipos) ---
ruta_doc = os.path.join(DATA_PATH, 'Indicadores de docentes y equipos de computo',
                        '03_Estudiantes_por_docente_y_equipo_2026.xlsx')
df_sede = pd.read_excel(ruta_doc, sheet_name='Detalle por sede')

# --- Info geografica para asignar comuna ---
ruta_geo = os.path.join(DATA_PATH, 'Informaci\u00f3n geogr\u00e1fica sedes.xlsx')
if not os.path.exists(ruta_geo):
    ruta_geo = os.path.join(DATA_PATH, 'info_geografica',
                            'Informaci\u00f3n geogr\u00e1fica_sedes_Educativas.xlsx')
df_geo = pd.read_excel(ruta_geo)

# Cruzar sede con comuna
df_geo_min = df_geo[['EeCodDane', 'EEComCor']].copy()
df_geo_min['EeCodDane'] = df_geo_min['EeCodDane'].astype(str).str.strip()
df_sede['cod_dane_sede'] = df_sede['cod_dane_sede'].astype(str).str.strip()
df_sc = df_sede.merge(df_geo_min, left_on='cod_dane_sede', right_on='EeCodDane', how='left')
df_sc = df_sc[df_sc['EEComCor'].between(1, 22)].copy()
df_sc['num_comuna'] = df_sc['EEComCor'].astype(int)

# Agregar por comuna
agg = df_sc.groupby('num_comuna').agg(
    matricula_oficial=('matricula', 'sum'),
    equipos_total=('equipos', 'sum'),
    est_equipo_prom=('estudiantes_por_equipo', 'mean'),
).reset_index()

# Tabla consolidada
df_edu = df_cu[['num_comuna', 'comuna', 'Total']].merge(agg, on='num_comuna', how='left')
df_edu.rename(columns={'Total': 'matricula_total'}, inplace=True)

# Indicadores municipales (constantes para todas las comunas)
df_edu['repitencia'] = 6.82  # Tasa municipal 2026
df_edu['est_por_docente'] = 24.64  # Ratio municipal 2026

print(f'Tabla de dimension educacion: {df_edu.shape}')
df_edu

---
## 4. Calcular scores normalizados por indicador

In [ ]:
# Aplicar normalizacion con refs fijos
df_edu['score_matricula'] = df_edu['matricula_total'].apply(
    lambda v: score_ref(v, **{k: REFS_EDUCACION['matricula'][k] for k in ['ref_min','ref_max','inverso']}))

df_edu['score_repitencia'] = df_edu['repitencia'].apply(
    lambda v: score_ref(v, **{k: REFS_EDUCACION['repitencia'][k] for k in ['ref_min','ref_max','inverso']}))

df_edu['score_est_docente'] = df_edu['est_por_docente'].apply(
    lambda v: score_ref(v, **{k: REFS_EDUCACION['est_por_docente'][k] for k in ['ref_min','ref_max','inverso']}))

df_edu['score_est_equipo'] = df_edu['est_equipo_prom'].apply(
    lambda v: score_ref(v, **{k: REFS_EDUCACION['est_por_equipo'][k] for k in ['ref_min','ref_max','inverso']}))

# Score de dimension (promedio simple de los 4 indicadores)
df_edu['score_educacion'] = (
    df_edu['score_matricula'] +
    df_edu['score_repitencia'] +
    df_edu['score_est_docente'] +
    df_edu['score_est_equipo']
) / 4

# Clasificacion
def clasificar(score):
    if score < 40: return 'Nivel 1 - Critico'
    elif score < 60: return 'Nivel 2 - En desarrollo'
    elif score < 80: return 'Nivel 3 - Adecuado'
    else: return 'Nivel 4 - Optimo'

df_edu['nivel'] = df_edu['score_educacion'].apply(clasificar)

print('Scores calculados:')
cols_show = ['comuna', 'score_matricula', 'score_repitencia',
             'score_est_docente', 'score_est_equipo', 'score_educacion', 'nivel']
print(df_edu[cols_show].to_string(index=False))

---
## 5. Visualizaciones del score de Educacion

In [ ]:
# Grafico 1: Score de dimension Educacion por comuna
fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#e78ac3' if s < 40 else '#fc8d62' if s < 60 else '#66c2a5' if s < 80 else '#8da0cb'
          for s in df_edu['score_educacion']]
bars = ax.bar(df_edu['comuna'], df_edu['score_educacion'], color=colors)
ax.axhline(40, color='red', linestyle='--', alpha=0.5, label='Critico (<40)')
ax.axhline(60, color='orange', linestyle='--', alpha=0.5, label='En desarrollo (<60)')
ax.axhline(80, color='green', linestyle='--', alpha=0.5, label='Adecuado (<80)')
ax.set_xlabel('Comuna')
ax.set_ylabel('Score Educacion (0-100)')
ax.set_title('Score Dimension Educacion y Desarrollo por comuna - ITT 2026')
ax.set_ylim(0, 100)
ax.legend(loc='upper right')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+1, f'{h:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 2: Desglose de scores por indicador (stacked horizontal)
fig, ax = plt.subplots(figsize=(12, 7))
comunas = df_edu['comuna']
y = range(len(comunas))

# Cada indicador contribuye 25% al score total
s1 = df_edu['score_matricula'] / 4
s2 = df_edu['score_repitencia'] / 4
s3 = df_edu['score_est_docente'] / 4
s4 = df_edu['score_est_equipo'] / 4

ax.barh(y, s1, label='Matricula', color='#66c2a5')
ax.barh(y, s2, left=s1, label='Repitencia', color='#fc8d62')
ax.barh(y, s3, left=s1+s2, label='Est/Docente', color='#8da0cb')
ax.barh(y, s4, left=s1+s2+s3, label='Est/Equipo', color='#e78ac3')

ax.set_yticks(y)
ax.set_yticklabels(comunas)
ax.set_xlabel('Contribucion al score (0-100)')
ax.set_title('Contribucion de cada indicador al Score Educacion')
ax.legend(loc='lower right')
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico 3: Ranking de comunas por score educacion
df_rank = df_edu[['comuna', 'score_educacion', 'nivel']].sort_values('score_educacion', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e78ac3' if s < 40 else '#fc8d62' if s < 60 else '#66c2a5' if s < 80 else '#8da0cb'
          for s in df_rank['score_educacion']]
bars = ax.barh(df_rank['comuna'], df_rank['score_educacion'], color=colors)
ax.axvline(40, color='red', linestyle='--', alpha=0.4)
ax.axvline(60, color='orange', linestyle='--', alpha=0.4)
ax.axvline(80, color='green', linestyle='--', alpha=0.4)
ax.set_xlabel('Score Educacion (0-100)')
ax.set_title('Ranking de comunas - Dimension Educacion ITT 2026')
ax.set_xlim(0, 100)
for bar in bars:
    w = bar.get_width()
    ax.text(w+0.5, bar.get_y()+bar.get_height()/2, f'{w:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Resumen y valor para el ITT

In [ ]:
# Resumen estadistico
print('=' * 60)
print('RESUMEN - DIMENSION EDUCACION Y DESARROLLO (ITT 2026)')
print('=' * 60)
print(f'\nScore promedio municipal: {df_edu["score_educacion"].mean():.1f}')
print(f'Score mediana:            {df_edu["score_educacion"].median():.1f}')
print(f'Score minimo:             {df_edu["score_educacion"].min():.1f} ({df_edu.loc[df_edu["score_educacion"].idxmin(), "comuna"]})')
print(f'Score maximo:             {df_edu["score_educacion"].max():.1f} ({df_edu.loc[df_edu["score_educacion"].idxmax(), "comuna"]})')
print(f'\nDistribucion por nivel:')
print(df_edu['nivel'].value_counts().to_string())
print(f'\n--- Valor para el ITT ---')
print(f'Score Educacion y Desarrollo (promedio 22 comunas): {df_edu["score_educacion"].mean():.1f}')
print(f'Este valor reemplaza el referente provisional de Pulmon de Oriente (54.9)')
print(f'\nNota: Repitencia y Est/Docente son constantes municipales.')
print(f'La variacion entre comunas se explica por Matricula y Est/Equipo.')

In [ ]:
# Tabla final exportable
print('\nTabla completa de scores por comuna:')
print('=' * 60)
cols_export = ['comuna', 'matricula_total', 'est_equipo_prom', 'repitencia',
               'est_por_docente', 'score_matricula', 'score_repitencia',
               'score_est_docente', 'score_est_equipo', 'score_educacion', 'nivel']
df_export = df_edu[cols_export].copy()
df_export['matricula_total'] = df_export['matricula_total'].apply(lambda x: f'{x:,.0f}')
df_export['est_equipo_prom'] = df_export['est_equipo_prom'].apply(lambda x: f'{x:.2f}')
df_export['score_educacion'] = df_export['score_educacion'].apply(lambda x: f'{x:.1f}')
print(df_export.to_string(index=False))

---
## 7. Notas metodologicas

### Indicadores usados y su estado:
| Indicador | Valor | Tipo | Fuente | Variacion por comuna |
|---|---|---|---|---|
| Matricula total | 5,313 - 25,902 | Positivo | 01_Matricula_2026 (Por comuna) | SI |
| Repitencia | 6.82% | Inverso | 02_Indicadores_2026 (municipal) | NO (constante) |
| Est/Docente | 24.64 | Inverso | 03_Estudiantes_2026 (municipal) | NO (constante) |
| Est/Equipo | Variable | Inverso | 03_Estudiantes_2026 (por sede) | SI |

### Limitaciones:
- Repitencia y Est/Docente son valores municipales aplicados a todas las comunas.
- La variacion real del score entre comunas depende solo de Matricula y Est/Equipo.
- Cuando se consiga desglose de repitencia y docentes por comuna, el score sera mas preciso.
- No se incluye desercion (dato no disponible en las fuentes actuales).
- No se incluye cobertura bruta/neta por comuna (requiere poblacion DANE por comuna).

### Refs utilizados:
| Indicador | ref_min | ref_max | Sustento |
|---|---|---|---|
| Matricula | 5,000 | 30,000 | Rango observado en 22 comunas (5,313 - 25,902) |
| Repitencia | 1.0% | 15.0% | Meta PD <5%, max historico ~12% |
| Est/Docente | 15 | 40 | Estandar MEN: 32 urbano, ideal <25 |
| Est/Equipo | 1 | 20 | Meta PD: 1 equipo cada 5 est (ratio 5) |

---
## 8. Mapa coropletico - Score Educacion por comuna

Visualizacion espacial del score de la dimension Educacion usando el poligono de comunas de Cali.

In [ ]:
# Instalar geopandas y folium si no estan (Colab)
try:
    import geopandas as gpd
    import folium
    from folium import Choropleth, GeoJsonTooltip
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'geopandas', 'folium', '-q'])
    import geopandas as gpd
    import folium
    from folium import Choropleth, GeoJsonTooltip

import zipfile
print('geopandas y folium listos')

In [ ]:
# Cargar poligonos de comunas
ruta_comunas_zip = os.path.join(DATA_PATH, 'info_geografica', 'Comunas.zip')

# Descomprimir si es necesario
carpeta_comunas = os.path.join(DATA_PATH, 'info_geografica', 'Comunas')
if not os.path.exists(carpeta_comunas):
    with zipfile.ZipFile(ruta_comunas_zip, 'r') as z:
        z.extractall(os.path.join(DATA_PATH, 'info_geografica'))
    print('Comunas.zip descomprimido')

# Buscar el shapefile o geojson dentro
archivos_comunas = []
for root, dirs, files in os.walk(carpeta_comunas):
    for f in files:
        if f.endswith(('.shp', '.geojson', '.json')):
            archivos_comunas.append(os.path.join(root, f))

if not archivos_comunas:
    # Buscar directamente en info_geografica
    for root, dirs, files in os.walk(os.path.join(DATA_PATH, 'info_geografica')):
        for f in files:
            if 'omuna' in f and f.endswith(('.shp', '.geojson', '.json')):
                archivos_comunas.append(os.path.join(root, f))

print(f'Archivos encontrados: {archivos_comunas}')
gdf_comunas = gpd.read_file(archivos_comunas[0])
print(f'Comunas cargadas: {len(gdf_comunas)}')
print(f'CRS: {gdf_comunas.crs}')
print(f'Columnas: {gdf_comunas.columns.tolist()}')
gdf_comunas.head()

In [ ]:
# Identificar columna de numero de comuna y preparar merge
# Buscar columna que tenga numeros de comuna (1-22)
col_comuna = None
for col in gdf_comunas.columns:
    if col == 'geometry':
        continue
    try:
        vals = pd.to_numeric(gdf_comunas[col], errors='coerce').dropna()
        if vals.between(1, 22).all() and len(vals) >= 20:
            col_comuna = col
            break
    except:
        pass

# Si no se encontro automaticamente, intentar por nombre
if col_comuna is None:
    for col in gdf_comunas.columns:
        if 'comuna' in col.lower() or 'com' in col.lower():
            col_comuna = col
            break

print(f'Columna de comuna identificada: {col_comuna}')
gdf_comunas['num_comuna'] = pd.to_numeric(gdf_comunas[col_comuna], errors='coerce').astype('Int64')

# Filtrar solo comunas urbanas
gdf_urb = gdf_comunas[gdf_comunas['num_comuna'].between(1, 22)].copy()
print(f'Comunas urbanas en shapefile: {len(gdf_urb)}')

# Merge con scores de educacion
gdf_edu = gdf_urb.merge(df_edu[['num_comuna', 'score_educacion', 'score_matricula',
                                 'score_est_equipo', 'nivel', 'matricula_total',
                                 'est_equipo_prom']],
                        on='num_comuna', how='left')
print(f'Comunas con score asignado: {gdf_edu["score_educacion"].notna().sum()}')

# Reproyectar a WGS84 para folium
gdf_edu = gdf_edu.to_crs(epsg=4326)
print('Reproyectado a EPSG:4326')

In [ ]:
# Mapa coropletico con matplotlib (estatico)
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
gdf_edu.plot(column='score_educacion', cmap='RdYlGn', linewidth=0.8,
             edgecolor='0.3', legend=True, ax=ax,
             legend_kwds={'label': 'Score Educacion (0-100)',
                          'orientation': 'horizontal',
                          'shrink': 0.6})

# Etiquetas de comuna
for idx, row in gdf_edu.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(f"C{int(row['num_comuna'])}\n{row['score_educacion']:.0f}",
                xy=(centroid.x, centroid.y), ha='center', fontsize=8,
                fontweight='bold')

ax.set_title('Score Dimension Educacion por Comuna - Cali 2026', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Mapa interactivo con Folium
# Centro de Cali
centro = [3.4516, -76.5320]
m = folium.Map(location=centro, zoom_start=12, tiles='CartoDB positron')

# Coropletico
folium.Choropleth(
    geo_data=gdf_edu.to_json(),
    data=df_edu,
    columns=['num_comuna', 'score_educacion'],
    key_on='feature.properties.num_comuna',
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name='Score Educacion (0-100)',
    nan_fill_color='gray'
).add_to(m)

# Tooltips con info
style_function = lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 0.5}
folium.GeoJson(
    gdf_edu.to_json(),
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['num_comuna', 'score_educacion', 'matricula_total', 'est_equipo_prom', 'nivel'],
        aliases=['Comuna:', 'Score Educacion:', 'Matricula Total:', 'Est/Equipo:', 'Nivel:'],
        localize=True
    )
).add_to(m)

print('Mapa interactivo generado. Pasa el mouse sobre cada comuna para ver detalles.')
m